# 06. 색 공간

BGR·RGB·HSV 변환, 이미지 합성, ROI와 마스크를 이용한 워터마크를 실습합니다.

강의 슬라이드의 코드를 실행 순서에 맞게 정리한 실습 노트북입니다.

> 이미지 예제는 노트북과 같은 위치에 `data` 폴더를 만들고 강의에서 사용하는 파일을 넣어 실행하세요.  
> `cv2.imshow()`와 카메라·마우스 예제는 데스크톱 Jupyter/VS Code 환경에서 실행하는 것을 권장합니다.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

def read_rgb(path):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


## BGR·RGB·HSV 변환


In [ ]:
image = read_rgb(Path("data/sample.jpg"))
hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image); axes[0].set_title("RGB")
for ax, channel, title in zip(axes[1:], cv2.split(hsv), ["Hue", "Saturation", "Value"]):
    ax.imshow(channel, cmap="gray"); ax.set_title(title)
for ax in axes: ax.axis("off")
plt.show()


## 크기를 맞춘 뒤 가중 합성


In [ ]:
overlay = read_rgb(Path("data/watermark_no_copy.png"))
overlay = cv2.resize(overlay, (image.shape[1], image.shape[0]))
blended = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)

plt.imshow(blended)
plt.axis("off");


## ROI와 마스크로 워터마크 합성


In [ ]:
base = image.copy()
logo = read_rgb(Path("data/watermark_no_copy.png"))
logo = cv2.resize(logo, (min(320, base.shape[1] // 3), min(180, base.shape[0] // 3)))

y0 = base.shape[0] - logo.shape[0]
x0 = base.shape[1] - logo.shape[1]
roi = base[y0:, x0:]

gray = cv2.cvtColor(logo, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(gray, 245, 255, cv2.THRESH_BINARY_INV)
background = cv2.bitwise_and(roi, roi, mask=cv2.bitwise_not(mask))
foreground = cv2.bitwise_and(logo, logo, mask=mask)
base[y0:, x0:] = cv2.add(background, foreground)

plt.imshow(base)
plt.axis("off");
